# 🍎 水果分类检测系统 - 核心代码演示

本 Notebook 演示了水果分类模型的核心流程：
1. 加载预训练模型
2. 图片预处理
3. 模型预测与结果展示
4. 批量测试与评估

## 1. 导入依赖

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from PIL import Image
from tensorflow import keras

# 设置中文字体
matplotlib.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

print("✅ 依赖导入成功")
print(f"  TensorFlow/Keras: {keras.__version__}")
print(f"  NumPy: {np.__version__}")

## 2. 定义常量与标签映射

In [ ]:
# 类别标签（与模型训练时一致）
LABELS = {
    0: "Apple",
    1: "Banana",
    2: "Mango",
    3: "Orange",
    4: "Pineapple",
}

LABEL_EMOJI = {
    "Apple": "🍎",
    "Banana": "🍌",
    "Mango": "🥭",
    "Orange": "🍊",
    "Pineapple": "🍍",
}

# 图片尺寸（与模型训练时一致）
IMG_SIZE = (128, 128)

# 路径配置
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
MODEL_PATH = os.path.join(BASE_DIR, "Model", "model.h5")
TEST_DIR = os.path.join(BASE_DIR, "Data", "test")

print(f"📂 项目根目录: {BASE_DIR}")
print(f"📂 模型路径: {MODEL_PATH}")
print(f"📂 测试数据路径: {TEST_DIR}")
print(f"🏷️  类别: {LABELS}")

## 3. 加载预训练模型

In [ ]:
# 加载模型
print("⏳ 正在加载模型 ...")
model = keras.models.load_model(MODEL_PATH)
print("✅ 模型加载成功!")
print(f"  模型结构摘要:")
model.summary()

## 4. 图片预处理函数

In [ ]:
def preprocess_image(image_path, target_size=IMG_SIZE):
    """
    加载并预处理图片
    
    Args:
        image_path: 图片文件路径
        target_size: 目标尺寸 (width, height)
    
    Returns:
        img_array: 预处理后的图像数组 (1, 128, 128, 3)
        original_img: 原始 PIL 图像对象
    """
    img = Image.open(image_path).convert("RGB")
    img_resized = img.resize(target_size)
    img_array = np.array(img_resized, dtype=np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # (1, 128, 128, 3)
    return img_array, img


def predict_image(model, image_path):
    """
    对单张图片进行分类预测
    
    Args:
        model: 训练好的 Keras 模型
        image_path: 图片文件路径
    
    Returns:
        result: 包含预测结果的字典
    """
    img_array, original_img = preprocess_image(image_path)
    
    # 预测
    predictions = model.predict(img_array, verbose=0)
    pred_class = int(np.argmax(predictions[0]))
    confidence = float(np.max(predictions[0])) * 100
    fruit_name = LABELS[pred_class]
    emoji = LABEL_EMOJI.get(fruit_name, "")
    
    result = {
        "fruit_name": fruit_name,
        "emoji": emoji,
        "confidence": confidence,
        "pred_class": pred_class,
        "predictions": predictions,
        "original_img": original_img,
    }
    return result


def display_prediction(result):
    """
    可视化显示预测结果（包括概率柱状图）
    """
    fruit_name = result["fruit_name"]
    emoji = result["emoji"]
    confidence = result["confidence"]
    predictions = result["predictions"]
    pred_class = result["pred_class"]
    original_img = result["original_img"]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ---- 左图：原图 + 预测结果 ----
    ax1 = axes[0]
    ax1.imshow(original_img)
    ax1.axis("off")
    ax1.set_title(
        f"{emoji}  {fruit_name}\n置信度: {confidence:.2f}%",
        fontsize=15, fontweight="bold", pad=12
    )
    
    # ---- 右图：概率柱状图 ----
    ax2 = axes[1]
    names = [f"{LABEL_EMOJI[LABELS[i]]} {LABELS[i]}" for i in range(len(LABELS))]
    probs = [float(predictions[0][i]) * 100 for i in range(len(LABELS))]
    
    colors = ["#27ae60" if i == pred_class else "#bdc3c7" for i in range(len(LABELS))]
    
    bars = ax2.barh(names[::-1], probs[::-1], color=colors[::-1], height=0.6)
    ax2.set_xlim(0, 105)
    ax2.set_xlabel("概率 (%)", fontsize=12)
    ax2.set_title("各类别概率分布", fontsize=14, fontweight="bold", pad=12)
    
    # 在柱状图上标注数值
    for bar, prob in zip(bars, probs[::-1]):
        ax2.text(
            prob + 1, bar.get_y() + bar.get_height() / 2,
            f"{prob:.1f}%", va="center", fontsize=11, fontweight="bold",
            color="#2c3e50"
        )
    
    ax2.tick_params(labelsize=11)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    return fig


print("✅ 工具函数定义完成")

## 5. 单张图片测试

选择一个测试类别进行演示。

In [ ]:
# 查看测试集中的可用类别和文件
for category in sorted(os.listdir(TEST_DIR)):
    cat_path = os.path.join(TEST_DIR, category)
    if os.path.isdir(cat_path):
        files = [f for f in os.listdir(cat_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
        print(f"{LABEL_EMOJI.get(category, '📁')} {category}: {len(files)} 张图片")

### 5.1 测试 Apple 🍎

In [ ]:
# 随机选择一张 Apple 图片进行测试
apple_dir = os.path.join(TEST_DIR, "Apple")
apple_images = [f for f in os.listdir(apple_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
if apple_images:
    test_img_path = os.path.join(apple_dir, apple_images[0])
    print(f"测试图片: {test_img_path}")
    result = predict_image(model, test_img_path)
    display_prediction(result)
else:
    print("❌ 未找到 Apple 测试图片")
</VOCode.Cell>
<VSCode.Cell language="markdown">
### 5.2 测试 Banana 🍌

In [ ]:
# 随机选择一张 Banana 图片进行测试
banana_dir = os.path.join(TEST_DIR, "Banana")
banana_images = [f for f in os.listdir(banana_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
if banana_images:
    test_img_path = os.path.join(banana_dir, banana_images[0])
    print(f"测试图片: {test_img_path}")
    result = predict_image(model, test_img_path)
    display_prediction(result)
else:
    print("❌ 未找到 Banana 测试图片")

### 5.3 测试 Mango 🥭

In [ ]:
# 随机选择一张 Mango 图片进行测试
mango_dir = os.path.join(TEST_DIR, "Mango")
mango_images = [f for f in os.listdir(mango_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
if mango_images:
    test_img_path = os.path.join(mango_dir, mango_images[0])
    print(f"测试图片: {test_img_path}")
    result = predict_image(model, test_img_path)
    display_prediction(result)
else:
    print("❌ 未找到 Mango 测试图片")

### 5.4 测试 Orange 🍊

In [ ]:
# 随机选择一张 Orange 图片进行测试
orange_dir = os.path.join(TEST_DIR, "Orange")
orange_images = [f for f in os.listdir(orange_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
if orange_images:
    test_img_path = os.path.join(orange_dir, orange_images[0])
    print(f"测试图片: {test_img_path}")
    result = predict_image(model, test_img_path)
    display_prediction(result)
else:
    print("❌ 未找到 Orange 测试图片")

### 5.5 测试 Pineapple 🍍

In [ ]:
# 随机选择一张 Pineapple 图片进行测试
pineapple_dir = os.path.join(TEST_DIR, "Pineapple")
pineapple_images = [f for f in os.listdir(pineapple_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
if pineapple_images:
    test_img_path = os.path.join(pineapple_dir, pineapple_images[0])
    print(f"测试图片: {test_img_path}")
    result = predict_image(model, test_img_path)
    display_prediction(result)
else:
    print("❌ 未找到 Pineapple 测试图片")

## 6. 批量测试与评估

对整个测试集进行分类，计算混淆矩阵和各类别准确率。

In [ ]:
import itertools
from sklearn.metrics import confusion_matrix, classification_report

def batch_evaluate(model, test_dir, target_size=IMG_SIZE):
    """
    批量评估模型在测试集上的表现
    
    Returns:
        y_true: 真实标签列表
        y_pred: 预测标签列表
        results: 每张图片的详细结果
    """
    y_true = []
    y_pred = []
    results = []
    
    # 类别名称到索引的映射
    label_to_idx = {name: idx for idx, name in LABELS.items()}
    
    for category in sorted(os.listdir(test_dir)):
        cat_path = os.path.join(test_dir, category)
        if not os.path.isdir(cat_path):
            continue
        
        true_label = label_to_idx.get(category)
        if true_label is None:
            continue
        
        image_files = [f for f in os.listdir(cat_path) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]
        
        for img_file in image_files:
            img_path = os.path.join(cat_path, img_file)
            try:
                img_array, _ = preprocess_image(img_path)
                preds = model.predict(img_array, verbose=0)
                pred_class = int(np.argmax(preds[0]))
                
                y_true.append(true_label)
                y_pred.append(pred_class)
                results.append({
                    "file": img_file,
                    "path": img_path,
                    "true_label": true_label,
                    "true_name": LABELS[true_label],
                    "pred_class": pred_class,
                    "pred_name": LABELS[pred_class],
                    "confidence": float(np.max(preds[0])) * 100,
                    "correct": true_label == pred_class,
                })
            except Exception as e:
                print(f"⚠️  处理 {img_path} 时出错: {e}")
    
    return y_true, y_pred, results


print("⏳ 正在批量评估测试集 ...")
y_true, y_pred, eval_results = batch_evaluate(model, TEST_DIR)
print(f"✅ 批量评估完成! 共测试 {len(eval_results)} 张图片")

### 6.1 分类报告

In [ ]:
# 输出分类报告
print("=" * 70)
print("📊  分类评估报告")
print("=" * 70)
print(classification_report(
    y_true, y_pred,
    target_names=[f"{LABEL_EMOJI[LABELS[i]]} {LABELS[i]}" for i in range(len(LABELS))],
    digits=4
))
print("=" * 70)

# 计算总体准确率
accuracy = np.sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
print(f"\n🎯  总体准确率: {accuracy:.2%} ({np.sum(np.array(y_true) == np.array(y_pred))}/{len(y_true)})")

### 6.2 混淆矩阵

In [ ]:
# 绘制混淆矩阵
cm = confusion_matrix(y_true, y_pred)
class_names = [f"{LABEL_EMOJI[LABELS[i]]}\n{LABELS[i]}" for i in range(len(LABELS))]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

ax.set(
    xticks=np.arange(len(class_names)),
    yticks=np.arange(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel="预测类别",
    ylabel="真实类别",
    title="混淆矩阵 (Confusion Matrix)",
)

# 在格子中标注数值
thresh = cm.max() / 2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(
        j, i, format(cm[i, j], "d"),
        ha="center", va="center",
        color="white" if cm[i, j] > thresh else "black",
        fontsize=14, fontweight="bold"
    )

plt.tight_layout()
plt.show()

### 6.3 各类别准确率对比

In [ ]:
# 计算各类别准确率
correct_per_class = {}
total_per_class = {}

for r in eval_results:
    name = r["true_name"]
    if name not in correct_per_class:
        correct_per_class[name] = 0
        total_per_class[name] = 0
    total_per_class[name] += 1
    if r["correct"]:
        correct_per_class[name] += 1

categories = list(LABELS.values())
accuracies = [correct_per_class[c] / total_per_class[c] * 100 for c in categories]
counts = [total_per_class[c] for c in categories]

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db", "#9b59b6"]
bars = ax.bar(categories, accuracies, color=colors_bar, width=0.6, edgecolor="white")

# 标注数值
for bar, acc_val, count in zip(bars, accuracies, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{acc_val:.1f}%\n({count} 张)",
        ha="center", va="bottom", fontsize=12, fontweight="bold"
    )

ax.set_ylim(0, 110)
ax.set_ylabel("准确率 (%)", fontsize=13)
ax.set_title("各类别识别准确率", fontsize=15, fontweight="bold", pad=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()

## 7. 自定义图片预测

你也可以使用自己的图片进行预测。将下面的 `custom_image_path` 替换为你的图片路径。

In [ ]:
# 自定义图片预测（将路径替换为你的图片）
custom_image_path = os.path.join(TEST_DIR, "Apple", "sample1.jpg")  # ← 修改这里

if os.path.exists(custom_image_path):
    print(f"🔍 正在预测: {custom_image_path}")
    result = predict_image(model, custom_image_path)
    display_prediction(result)
else:
    print(f"❌ 文件不存在: {custom_image_path}")
    print("💡 请修改 custom_image_path 变量，指向你自己的图片文件")